# 실습 10: 드문 쪽을 놓치지 않게 만들기
- 상황: 어제 모델은 실제 불량 스물한 건 중 두 건만 잡았다
- 목표: 놓친 쪽을 줄이는 방법을 적용하고, 무엇을 내줬는지 함께 적는다

## Step 0. 어제 상태까지 재현하기

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

스케일러 = StandardScaler()
X_train_스케일 = 스케일러.fit_transform(X_train)
X_test_스케일 = 스케일러.transform(X_test)

로지스틱모델 = LogisticRegression()
로지스틱모델.fit(X_train_스케일, y_train)
예측 = 로지스틱모델.predict(X_test_스케일)

정확도 = accuracy_score(y_test, 예측)
불량예측건수 = (예측 == 1).sum()
실제로_불량이었던_건수 = ((예측 == 1) & (y_test == 1)).sum()

print("학습용:", X_train.shape, "불량 건수:", y_train.sum())
print("시험용:", X_test.shape, "불량 건수:", y_test.sum())
print("1. 정확도:", round(정확도 * 100, 2), "%")
print("2. 불량이라고 예측한 건수:", 불량예측건수)
print("3. 그중 실제로 불량이었던 건수:", 실제로_불량이었던_건수)


학습용: (1253, 50) 불량 건수: 83
시험용: (314, 50) 불량 건수: 21
1. 정확도: 92.99 %
2. 불량이라고 예측한 건수: 3
3. 그중 실제로 불량이었던 건수: 1


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 드문 쪽을 다루는 말

| 말 | 뜻 |
|---|---|
| 클래스 불균형 | 한쪽이 지나치게 드문 상태. 우리 데이터는 불량이 약 6.6%뿐이다 |
| 클래스 가중치 | 드문 쪽 한 건을 여러 건만큼 무겁게 세도록 모델에 알려주는 설정 |
| 언더샘플링 | 많은 쪽을 줄여서 양쪽 수를 맞추는 방법. 데이터를 버리게 된다 |
| 오버샘플링 | 드문 쪽을 늘려서 양쪽 수를 맞추는 방법. 없던 기록을 만들어 넣게 된다 |
| 재현율 | 실제 불량 중 몇 %를 잡았나. 오늘 올리려는 숫자 |
| 정밀도 | 불량이라 한 것 중 몇 %가 진짜였나. 오늘 내주게 될 숫자 |

## Step 2. 무게를 다르게 주기

In [2]:
# 표준화와 모델을 한 줄로 묶어주는 도구들을 불러온다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# class_weight="balanced" - 드문 쪽 한 건을 그만큼 무겁게 세라는 뜻. 오늘 추가한 것은 이 한 조각뿐이다
가중치모델 = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

# 학습용으로만 학습시킨다. 시험용은 여전히 건드리지 않는다
가중치모델.fit(X_train, y_train)

# 시험용 입력만 넣어 답을 받는다
가중치예측 = 가중치모델.predict(X_test)

print("불량이라고 예측한 건수:", 가중치예측.sum())
print("그중 진짜 불량:", ((가중치예측 == 1) & (y_test == 1)).sum())


불량이라고 예측한 건수: 77
그중 진짜 불량: 10


어제(가중치 없음) 대비 불량 21건 중 잡은 건수가 1건→10건으로 늘었지만, 불량이라 예측한 77건 중 실제 불량은 10건뿐이라 정밀도는 낮아졌습니다(재현율은 오르고 정밀도는 내준 전형적인 트레이드오프).

### 문법 노트 - 오늘 추가한 한 조각

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| class_weight="balanced" | 적은 쪽 한 건을 더 무겁게 세게 한다 | 그냥 두면 모델이 많은 쪽만 맞히고 만족해버린다 |
| max_iter=1000 | 답을 찾을 때까지 계산을 더 오래 하게 둔다 | 기본값으로는 다 못 찾고 멈췄다는 경고가 뜬다 |

## Step 3. 전후 숫자 비교하기

In [3]:
# 네 칸 표와 지표들을 계산해주는 도구를 불러온다
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 어제 예측과 오늘 예측을 나란히 놓고 같은 자로 잰다
for 이름, 예측값 in [("손 안 댐", 예측), ("가중치", 가중치예측)]:
    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측값).ravel()
    print(f"[{이름}]")
    print("  정확도:", round((예측값 == y_test).mean() * 100, 2), "%")
    print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
    print("  재현율:", round(recall_score(y_test, 예측값), 3),
          "정밀도:", round(precision_score(y_test, 예측값, zero_division=0), 3),
          "F1:", round(f1_score(y_test, 예측값), 3))


[손 안 댐]
  정확도: 92.99 %
  잡은 불량: 1 / 놓친 불량: 20 / 헛경보: 2
  재현율: 0.048 정밀도: 0.333 F1: 0.083
[가중치]
  정확도: 75.16 %
  잡은 불량: 10 / 놓친 불량: 11 / 헛경보: 67
  재현율: 0.476 정밀도: 0.13 F1: 0.204


[손 안 댐 (어제)] 정확도 92.99% / 잡은 불량 1, 놓친 20, 헛경보 2 / 재현율 0.048, 정밀도 0.333, F1 0.083

[가중치 (오늘)] 정확도 75.16% / 잡은 불량 10, 놓친 11, 헛경보 67 / 재현율 0.476, 정밀도 0.13, F1 0.204

재현율은 약 10배 올랐지만(0.048→0.476), 정밀도가 크게 내려가고(0.333→0.13) 헛경보가 2건→67건으로 늘면서 전체 정확도도 떨어졌습니다 — F1은 그래도 0.083→0.204로 개선.

## Step 4. 많은 쪽을 줄여서 해보기

In [4]:
# 학습용 안에서 불량과 양품을 따로 떼어낸다
불량_train = X_train[y_train == 1]
양품_train = X_train[y_train == 0]

# 양품에서 불량 건수만큼만 무작위로 뽑는다 (시험용은 전혀 건드리지 않는다)
양품_축소 = 양품_train.sample(n=len(불량_train), random_state=42)

X_train_균형 = pd.concat([불량_train, 양품_축소])
y_train_균형 = pd.concat([y_train.loc[불량_train.index], y_train.loc[양품_축소.index]])

# 표준화와 로지스틱 회귀를 함께 학습시킨다
언더샘플모델 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
언더샘플모델.fit(X_train_균형, y_train_균형)

# 원래 시험용(X_test) 그대로 예측한다
언더샘플예측 = 언더샘플모델.predict(X_test)

print("1. 줄인 학습용 건수:", len(X_train_균형),
      "(불량", (y_train_균형 == 1).sum(), "/ 양품", (y_train_균형 == 0).sum(), ")")

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 언더샘플예측).ravel()
print("2. 정확도:", round((언더샘플예측 == y_test).mean() * 100, 2), "%")
print("3. 잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("4. 재현율:", round(recall_score(y_test, 언더샘플예측), 3),
      "정밀도:", round(precision_score(y_test, 언더샘플예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 언더샘플예측), 3))


1. 줄인 학습용 건수: 166 (불량 83 / 양품 83 )
2. 정확도: 70.38 %
3. 잡은 불량: 15 / 놓친 불량: 6 / 헛경보: 87
4. 재현율: 0.714 정밀도: 0.147 F1: 0.244


가중치 방식(재현율 0.476, F1 0.204)보다 재현율과 F1이 더 올랐지만, 정확도는 더 떨어지고 헛경보는 더 늘었습니다 — 데이터를 버려서(83건씩만 사용) 학습량이 확 줄어든 대가입니다.

## Step 5. 전후 비교표

| 처리 | 정확도 | 잡은 불량 | 놓친 불량 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|---|
| 손 안 댐 | [92.99]% | [1] | [20] | [2] | [0.048] | [0.333] | [0.083] |
| 가중치 주기 | [75.16]% | [10] | [11] | [67] | [0.476] | [0.13] | [0.204] |
| 많은 쪽 줄이기 | [70.38]% | [15] | [6] | [87] | [0.714] | [0.147] | [0.244] |

## Step 6. 정직한 처리의 선

- 세 방법 모두 **학습용에만** 적용했다
- 시험용 314건은 처음 나눈 그대로 두었다 (불량 21건 그대로)
- 시험용을 손보면 점수는 올라가지만, 현장에 나가는 순간 그 점수는 없다

---
## 직접 해보기 (도전) - 시험지까지 손대면 어떻게 되나

- 상황: 학습용에만 손대라고 했는데, 시험용에도 손대면 점수가 어떻게 나올까
- 할 일: 시험용을 반반으로 맞춰놓고 같은 모델의 점수를 다시 잰다
- 결과물: 두 줄짜리 비교표 1개

In [5]:
# 원래 시험용에서 불량과 양품을 따로 떼어낸다 (X_test, y_test 자체는 건드리지 않는다)
불량_test = X_test[y_test == 1]
양품_test = X_test[y_test == 0]

# 양품에서 불량 건수만큼만 무작위로 뽑아 반반짜리 시험용을 새 이름으로 만든다
양품_test_축소 = 양품_test.sample(n=len(불량_test), random_state=42)

X_test_반반 = pd.concat([불량_test, 양품_test_축소])
y_test_반반 = pd.concat([y_test.loc[불량_test.index], y_test.loc[양품_test_축소.index]])

# Step 2에서 학습시킨 가중치모델은 다시 학습시키지 않고 그대로 채점에만 쓴다
예측_반반 = 가중치모델.predict(X_test_반반)

def 요약행(이름, y_참, 예측값):
    return {
        "시험용": 이름,
        "건수": len(y_참),
        "정확도(%)": round((예측값 == y_참).mean() * 100, 2),
        "재현율": round(recall_score(y_참, 예측값), 3),
        "정밀도": round(precision_score(y_참, 예측값, zero_division=0), 3),
        "F1": round(f1_score(y_참, 예측값), 3),
    }

비교표_시험용 = pd.DataFrame([
    요약행("원래 시험용 (314건)", y_test, 가중치예측),
    요약행("반반 시험용", y_test_반반, 예측_반반),
])
비교표_시험용


,시험용,건수,정확도(%),재현율,정밀도,F1
0,원래 시험용 (314건),314,75.16,0.476,0.130,0.204
1,반반 시험용,42,66.67,0.476,0.769,0.588


재현율은 똑같습니다(같은 모델·같은 임계값이니 불량 21건 중 잡는 비율은 안 변함). 하지만 시험용의 양품을 293건→21건으로 줄이니 헛경보가 줄어든 것처럼 보여서 정밀도(0.130→0.769)와 F1(0.204→0.588)이 크게 좋아 보입니다 — 모델은 그대로인데 시험지 구성만 바꿔서 점수가 부풀려진 것입니다.

### 시험지를 손대면

| 채점 방식 | 건수 | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|
| 원래 시험용 | [314] | [74.84]% | [0.476] | [0.128] | [0.202] |
| 반반으로 맞춘 시험용 | [42] | [69.05]% | [0.476] | [0.833] | [0.606] |